In [ ]:
# this notebook is for counting the inputs tokens for all the 4 granularities, 
# to have an estimate of the cost of the experiments.

# conclusion: too expensive to use the open-ai models (at least the good ones)

In [41]:
from tqdm.notebook import tqdm

from pathlib import Path
import pandas as pd


GRANURALITIES = ["file", "class", "method", "block"]
DATA_FOLDER_PATH = "./data/"

code_snippets = {}

# Iterates over the directory and prints only files
for granularity in GRANURALITIES:
    folder_path = f"{DATA_FOLDER_PATH}/{granularity}"
    path = Path(folder_path)
    projects_file_names = [f.name for f in path.iterdir() if f.is_file()]

    code_snippets[granularity] = {}
    for project_file_name in projects_file_names:
        project_name = project_file_name.split("-")[0]
        
        code_snippets[granularity][project_name] = pd.read_csv(f"{folder_path}/{project_file_name}")
    

PROJECTS_NAMES = list(code_snippets[GRANURALITIES[0]].keys())

In [42]:
example_df = code_snippets["file"][PROJECTS_NAMES[0]]

ALL_COMMENTS_COLUMNS = [col for col in example_df.columns if "comment" in col.lower() and "each" not in col.lower() and type(example_df[col][0]) == str]

# verify that all granularities have the same comment columns
for granularity in GRANURALITIES:
    example_df = code_snippets[granularity][PROJECTS_NAMES[0]]
    all_comments_columns = [col for col in example_df.columns if "comment" in col.lower() and "each" not in col.lower() and type(example_df[col][0]) == str]
    assert all_comments_columns == ALL_COMMENTS_COLUMNS, f"Granularity {granularity} has different comment columns: {all_comments_columns} != {ALL_COMMENTS_COLUMNS}"


In [43]:
def remove_comments(code : str, comments_set : str) -> str:
    for comment in comments_set:
        if comment.strip() != "":
            code = code.replace(comment, "")
    
    return code

In [44]:

for granularity in tqdm(GRANURALITIES, desc="granularity", position=0):
    for project_name in tqdm(PROJECTS_NAMES, desc="project", position=1, leave=False):
        df = code_snippets[granularity][project_name]
        df["code_without_comments"] = df.apply(lambda row: remove_comments(row["Content"], row[ALL_COMMENTS_COLUMNS]), axis=1)

granularity:   0%|          | 0/4 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

In [45]:
import tiktoken

MODEL_NAME = "gpt-5"  # Replace with your model name

# Automatically retrieves the correct encoding (e.g., o200k_base for gpt-4o)
encoding = tiktoken.encoding_for_model(MODEL_NAME)

def count_tokens(text: str) -> int:
    
    # Encode the text into integer token IDs
    token_list = encoding.encode(text)
    
    # The number of integers in the list is your token count
    return len(token_list)

text_to_check = "Counting tokens with tiktoken is incredibly fast! 🚀"

print(f"Token Count: {count_tokens(text_to_check)}")

Token Count: 12


In [46]:
count_tokens("yes"), count_tokens("no")

(1, 1)

In [47]:

code_column = "code_without_comments"


for granularity in tqdm(GRANURALITIES, desc="granularity", position=0):
    for project_name in tqdm(PROJECTS_NAMES, desc="project", position=1, leave=False):
        code_df = code_snippets[granularity][project_name]
        code_df["token_count"] = code_df[code_column].apply(count_tokens)
        # store back if you want to ensure the dict holds the updated df (usually not needed)
        code_snippets[granularity][project_name] = code_df




granularity:   0%|          | 0/4 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

project:   0%|          | 0/18 [00:00<?, ?it/s]

In [48]:
granularity = "file"
project_name = PROJECTS_NAMES[0]

code_df = code_snippets[granularity][project_name]

code_df.token_count.describe()

count     1498.00000
mean      2168.12283
std       4043.65019
min         12.00000
25%        418.25000
50%        919.50000
75%       2336.25000
max      91208.00000
Name: token_count, dtype: float64

In [ ]:
total_input_tokens = 0

for granularity in GRANURALITIES:
    for project_name in PROJECTS_NAMES:
        code_df = code_snippets[granularity][project_name]
        total_input_tokens += code_df.token_count.sum() + 60 * code_df.shape[0] # 200 is the number of tokens in the prompt without the code snippet, which is the same for all rows

total_output_tokens = 2 * sum([df.shape[0] for granularity in GRANURALITIES for project_name in PROJECTS_NAMES for df in [code_snippets[granularity][project_name]]])

print(f"Total input tokens across all granularities and projects: {total_input_tokens}") # 176131675
print(f"Total output tokens across all granularities and projects: {total_output_tokens}")


In [50]:
import json

with open("gpt_pricing.json", "r") as f:
    gpt_pricing = json.load(f)

In [51]:
model_price_df = pd.DataFrame(columns=['model', 'experiment cost (USD)'])

# all of our prompts will be less than 272.000 tokens
# so we can use short context pricing for all the LLM calls
context_size = "short_context"

for pricing_info in gpt_pricing:
    model = pricing_info["model"]

    prices = pricing_info[context_size]

    input_price_per_1M_tokens = float(prices["input"][1:])
    output_price_per_1M_tokens = float(prices["output"][1:])

    input_cost = (total_input_tokens / 1_000_000) * input_price_per_1M_tokens
    output_cost = (total_output_tokens / 1_000_000) * output_price_per_1M_tokens

    experiment_cost = input_cost + output_cost

    print(f"Model: {model}")
    # print(f"Total input price: {input_cost:.2f} USD")
    # print(f"Total output price: {output_cost:.2f} USD")
    print(f"Total experiment cost: \033[1m{experiment_cost:.2f} USD\033[0m")
    print("-" * 30)


    model_price_df.loc[len(model_price_df)] = [model, f"$ {round(experiment_cost, 2)}"]

print('#' * 30 + 'Other models pricing: ')


with open("other_models_pricing.json", "r") as f:
    other_pricing = json.load(f)

for pricing_info in other_pricing:
    model = pricing_info["Model"]

    prices = pricing_info

    input_price_per_1M_tokens = prices["Input"]
    output_price_per_1M_tokens = prices["Output"]

    input_cost = (total_input_tokens / 1_000_000) * input_price_per_1M_tokens
    output_cost = (total_output_tokens / 1_000_000) * output_price_per_1M_tokens

    experiment_cost = input_cost + output_cost

    print(f"Model: {model}")
    # print(f"Total input price: {input_cost:.2f} USD")
    # print(f"Total output price: {output_cost:.2f} USD")
    print(f"Total experiment cost: \033[1m{experiment_cost:.2f} USD\033[0m")
    print("-" * 30)

    model_price_df.loc[len(model_price_df)] = [model, f"$ {round(experiment_cost, 2)}"]


Model: gpt-5.5
Total experiment cost: 1012.03 USD
------------------------------
Model: gpt-5.5-pro
Total experiment cost: 6072.20 USD
------------------------------
Model: gpt-5.4
Total experiment cost: 506.02 USD
------------------------------
Model: gpt-5.4-mini
Total experiment cost: 151.81 USD
------------------------------
Model: gpt-5.4-nano
Total experiment cost: 40.52 USD
------------------------------
Model: gpt-5.4-pro
Total experiment cost: 6072.20 USD
------------------------------
##############################Other models pricing: 
Model: gpt-5.2
Total experiment cost: 356.77 USD
------------------------------
Model: gpt-5.2-pro
Total experiment cost: 4281.20 USD
------------------------------
Model: gpt-5.1
Total experiment cost: 254.83 USD
------------------------------
Model: gpt-5
Total experiment cost: 254.83 USD
------------------------------
Model: gpt-5-mini
Total experiment cost: 50.97 USD
------------------------------
Model: gpt-5-nano
Total experiment cost: 1

In [52]:
model_price_df.to_csv("model_price_estimation.csv", index=False)

In [53]:
model_price_df

,model,experiment cost (USD)
0,gpt-5.5,$ 1012.03
1,gpt-5.5-pro,$ 6072.2
2,gpt-5.4,$ 506.02
3,gpt-5.4-mini,$ 151.81
4,gpt-5.4-nano,$ 40.52
5,gpt-5.4-pro,$ 6072.2
6,gpt-5.2,$ 356.77
7,gpt-5.2-pro,$ 4281.2
8,gpt-5.1,$ 254.83
9,gpt-5,$ 254.83


In [54]:
round(3.14159, 2)

3.14

In [55]:
model_price_df

,model,experiment cost (USD)
0,gpt-5.5,$ 1012.03
1,gpt-5.5-pro,$ 6072.2
2,gpt-5.4,$ 506.02
3,gpt-5.4-mini,$ 151.81
4,gpt-5.4-nano,$ 40.52
5,gpt-5.4-pro,$ 6072.2
6,gpt-5.2,$ 356.77
7,gpt-5.2-pro,$ 4281.2
8,gpt-5.1,$ 254.83
9,gpt-5,$ 254.83


In [56]:
prompt = """
Act as an Technical Debt detector specialist. Tell if the following code has Technical Debt or not.
Only answer "yes" or "no".

Code: {"""

In [57]:
count_tokens(prompt)

32